# Бенчмарк цензор-модуля — метрики и интерактивные графики

Тонкая обёртка над `censor_guard.benchmark.run_benchmark`. Вся логика (загрузка
датасета, метрики, графики, отчёт) живёт в пакете `censor_guard/benchmark/`.
Здесь — запуск и inline-просмотр интерактивных Plotly-графиков.

Качество меряется на курируемом датасете
[`vekshinkir/image-censorship-small`](https://huggingface.co/datasets/vekshinkir/image-censorship-small),
split **`benchmarking`** (690 размеченных картинок: safe/unsafe, категория,
источник, AI-генерация, adversarial/edge-case флаги).

То же самое из консоли одной командой:

```
python -m censor_guard.benchmark            # весь split benchmarking
python -m censor_guard.benchmark --limit 60 # быстрая проба
```

Артефакты прогона (`save_report=True`) складываются в
`reports/benchmark_<timestamp>/`: самодостаточный `dashboard.html`,
`report.md`, `predictions.csv`, `metrics.json`.

In [ ]:
import os, sys
from pathlib import Path

# Запуск из папки notebooks/ — добавляем корень проекта в путь.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")

# Сколько картинок прогонять (None → весь split benchmarking = 690).
LIMIT = int(os.environ["BENCH_LIMIT"]) if os.environ.get("BENCH_LIMIT") else None
LIMIT

## Запуск

Загрузка датасета → прогрев моделей → классификация (с прогресс-баром) → метрики.
`save_report=True` сложит `reports/benchmark_<timestamp>/` со всеми артефактами,
включая интерактивный `dashboard.html`.

In [ ]:
from censor_guard.benchmark import run_benchmark, show_figures

result = run_benchmark(
    split="benchmarking",
    limit=LIMIT,        # None → все 690 картинок
    save_report=True,
)
print("Отчёт сохранён в:", result["report_dir"])

## Интерактивные графики inline

`show_figures` показывает те же Plotly-фигуры, что встроены в `dashboard.html`:
обзор (матрица ошибок + распределение оценок), ROC по категориям, метрики по
категориям, срезы по источникам и AI-генерации, латентность. Графики
интерактивны — zoom, hover, скрытие серий по клику в легенде.

In [ ]:
show_figures(result["figures"])

## Таблицы метрик

Сырые предсказания и сводные метрики доступны прямо в `result`.

In [ ]:
import pandas as pd
from IPython.display import display

print("Общие метрики (без adversarial/edge):")
display(pd.DataFrame([result["overall"]]))

print("По категориям таксономии:")
display(pd.DataFrame(result["per_category"]).T)

print("По источникам данных:")
display(pd.DataFrame(result["per_dataset"]))

print("AI-генерация vs реальные:")
display(pd.DataFrame(result["by_ai_generated"]))

print("Устойчивость (adversarial / edge-case):")
display(pd.DataFrame([{**result["adversarial"], **{f"edge_{k}": v for k, v in result["edge_case"].items()}}]))

print("Латентность:")
display(pd.DataFrame([result["latency"]]))